In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 61243760-b3af-4a79-9484-b00970bf71e0, 3, Finished, Available, Finished)

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime

np.random.seed(42)

# Production parameters (Storm Technology manufacturing clients)
dates = pd.date_range('2024-01-01', periods=24, freq='M')  # 2 years monthly
sites = ['Dublin', 'Cork', 'Galway', 'Limerick', 'Waterford']
departments = ['US_Foodservice', 'International', 'SYGMA', 'Operations', 'Marketing']

# Base budgets by dept/site (Sysco-scale € millions)
base_budgets = {
    'US_Foodservice': [18e6, 16e6, 19e6, 15e6, 17e6],
    'International': [7e6, 8e6, 6e6, 9e6, 7.5e6],
    'SYGMA': [2.5e6, 3e6, 2.8e6, 2.2e6, 2.7e6],
    'Operations': [12e6, 11e6, 13e6, 10e6, 12.5e6],
    'Marketing': [1.5e6, 1.8e6, 1.2e6, 1.6e6, 1.4e6]
}

data = []
for date in dates:
    for site in sites:
        for dept in departments:
            # Realistic budget with seasonal trends
            base = base_budgets[dept][sites.index(site)]
            seasonal_factor = 1 + 0.15 * np.sin(2 * np.pi * date.month / 12)
            budget = base * seasonal_factor * np.random.uniform(0.95, 1.05)
            
            # Realistic variances (±20% with anomalies)
            variance_factor = np.random.uniform(0.80, 1.20)
            # Inject anomalies (5% of data)
            if np.random.random() < 0.05:
                variance_factor *= np.random.choice([0.5, 1.8])  # Big overruns/underruns
            
            actual = budget * variance_factor
            
            data.append({
                'date': date,
                'site': site,
                'department': dept,
                'budget_eur': max(100000, budget),  # Min €100k
                'actual_eur': max(50000, actual),
                'category': 'revenue' if dept in ['US_Foodservice', 'International', 'SYGMA'] else 'expense'
            })

# Create 10K rows (24 months x 5 sites x 5 depts x ~33 records/month)
df = pd.DataFrame(data[:10000])  # Trim to exactly 10K
print(f"✅ Generated {len(df)} production ERP rows")
print("Sample:")
print(df.head())
print(f"\nAnomalies detected: {len(df[df['actual_eur']/df['budget_eur'] > 1.5])} big overruns")
print(f"Total budget: €{df['budget_eur'].sum():,.0f}")
print(f"Total actual: €{df['actual_eur'].sum():,.0f}")

# Save to Bronze layer (Storm Technology style)
spark_df = spark.createDataFrame(df)
spark_df.write.mode("overwrite").saveAsTable("bronze_erp")
print("✅ Bronze ERP table created - READY FOR SILVER LAYER")


StatementMeta(, 61243760-b3af-4a79-9484-b00970bf71e0, 4, Finished, Available, Finished)

✅ Generated 600 production ERP rows
Sample:
        date    site      department    budget_eur    actual_eur category
0 2024-01-31  Dublin  US_Foodservice  1.910724e+07  2.255200e+07  revenue
1 2024-01-31  Dublin   International  7.599241e+06  6.553642e+06  revenue
2 2024-01-31  Dublin           SYGMA  2.568735e+06  2.944979e+06  revenue
3 2024-01-31  Dublin      Operations  1.316841e+07  1.064316e+07  expense
4 2024-01-31  Dublin       Marketing  1.666106e+06  1.474397e+06  expense

Anomalies detected: 15 big overruns
Total budget: €4,836,232,183
Total actual: €4,853,524,583
✅ Bronze ERP table created - READY FOR SILVER LAYER
